# TM-Ca ensemble -- PCA / t-SNE / UMAP + GMM/HDBSCAN clustering, pooled across all 6 ABCfold backends

**Kernel:** `abcfold-npf-notebook` (`envs/notebook.yaml`)

Single notebook, not split by ligand category like `AF3_NPF_pipeline/notebook/tm_conformation_clustering_{gibberellin,nitrate,other_ligand,apoform}.ipynb`
-- ABCfold's 6-backend runs (AlphaFold3, Boltz-2, Chai-1, OpenFold3,
Protenix, RosettaFold3) are much more expensive than AF3 alone, so results
land one protein x form at a time rather than in one big batch; add a new
markdown + code cell pair to the "Per-protein cells" section below as each
protein's `worflows/postprocessing/Snakefile` (stage 6,
`scripts/tm_helix_alignment.py`) output appears under `results/tm_alignment/`.

`scripts/tm_helix_alignment.py` already pools every backend's CIFs (across
every seed x diffusion/sample) into one `results/tm_alignment/<protein>__{apo,holo}/`
ensemble per run, tagging each frame with a `model` column
(`alphafold3`/`boltz`/`chai1`/`openfold3`/`protenix`/`rosettafold3`) -- so
unlike `AF3_NPF_pipeline`'s notebooks (AF3 only, `color_by="status"` was
the only interesting axis, and a separate `_boltz` fork was needed just to
overlay a second model), **`color_by="model"` is the key new axis here**
and every `plot_*` function below defaults to it: a single backend's
diffusion doesn't always recover every conformation a second one finds
(that's the whole reason this pipeline exists instead of extending
`AF3_NPF_pipeline`).

```python
plot_pca(protein)                                 # colour by model (default) -- which of the 6 backends produced each point
plot_pca(protein, color_by='status')              # colour by status instead (apo/holo)
plot_pca(protein, color_by='rmsd_tm')             # colour by per-frame RMSD (A) to the converged TM-helix mean

plot_umap(protein, n_clusters=3)                                  # GMM manual: fit GMM-3
plot_umap(protein, n_clusters='auto')                             # GMM auto: BIC-knee sweep k=1..20

plot_umap(protein, cluster_method='hdbscan', n_clusters='auto')   # HDBSCAN auto: Optuna/DBCV search
plot_umap(protein, cluster_method='hdbscan', n_clusters='manual', # HDBSCAN manual: explicit params
          hdbscan_min_cluster_size=15)
```

**A caveat worth watching for, visible via `color_by='model'`:** backends
don't all contribute the same number of frames per protein -- e.g.
OpenFold3 currently produces far more samples per seed than the other 5
backends for the one protein this notebook currently covers, which can
dominate a joint PCA/UMAP fit or GMM/HDBSCAN clustering pass by sheer
point count. Worth keeping in mind when reading a cluster's backend
composition, not something this notebook corrects for.

Another one: RosettaFold3 writes both a `..._model.cif` and a
`..._model_fixed.cif` per (seed, sample) -- two distinct CIFs with
near-identical coordinates that `scripts/tm_helix_alignment.py`'s
`parse_frame_id()` doesn't distinguish (its regex only captures
seed/sample), so both get pooled as separate frames sharing one
`frame_id` -- effectively near-duplicating RosettaFold3's weight in this
ensemble. `_reannotate` below tolerates the resulting symlink-name
collision (skips the second one rather than erroring) but doesn't
deduplicate the underlying frames.

Also worth noting: `find_confidence()` in `scripts/tm_helix_alignment.py`
now does a real per-backend pTM/ipTM lookup (each of the 6 backends writes
its confidence JSON/npz under a different name/layout -- see that function
for the per-backend mapping), so `color_by='ptm'` and `color_by='iptm'` both
work. One asymmetry to expect on apoform (single-chain) runs: AlphaFold3
reports iptm as null (no interface to score) -- correctly NaN here -- while
the other 5 backends report 0.0 for the same case instead.

**Prerequisite:** run `worflows/postprocessing/Snakefile` (stage 6,
`scripts/tm_helix_alignment.py`) first for every protein you want to look
at here -- this notebook only reads `results/tm_alignment/`, it does not
compute alignments or touch `results/abcfold/` directly (except to
rediscover CIFs for reannotation symlinks, see `_reannotate` in the setup
cell).


In [1]:
from pathlib import Path

import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import umap
import optuna
from hdbscan.validity import validity_index
from kneed import KneeLocator
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture

optuna.logging.set_verbosity(optuna.logging.WARNING)  # one INFO line per trial is too noisy at hdbscan_n_trials=40/protein

ROOT             = Path("..")
ABCFOLD_OUT_ROOT = ROOT / "results" / "abcfold"
ALIGN_ROOT       = ROOT / "results" / "tm_alignment"
REANN_ROOT       = ROOT / "results" / "tm_reannotated"
FIG_ROOT         = ROOT / "results" / "figures"

GMM_PALETTE = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3",
    "#ff7f00", "#a65628", "#f781bf", "#999999",
]

# Fixed colours for the categorical color_by="status" scatter (apo vs holo)
STATUS_PALETTE = {"apo": "#7f7f7f", "holo": "#d62728"}

# Fixed colours for the categorical color_by="model" scatter -- the new
# axis this ABCfold pipeline exists for (AF3_NPF_pipeline's single-backend
# notebooks only ever had "status" to colour by; every protein here pools
# up to 6 backends' frames instead of one, see _load_protein below).
# "unknown" is _backend_of()'s fallback for a CIF path that doesn't match
# any BACKEND_PATTERNS entry -- shouldn't happen in practice, kept for
# safety since it's cheap to render if it ever does.
MODEL_PALETTE = {
    "alphafold3":   "#1f77b4",
    "boltz":        "#ff7f00",
    "chai1":        "#2ca02c",
    "openfold3":    "#d62728",
    "protenix":     "#9467bd",
    "rosettafold3": "#8c564b",
    "unknown":      "#7f7f7f",
}

# color_by name -> (palette dict, category display/legend order)
CATEGORICAL_COLOR_CONFIG = {
    "status": (STATUS_PALETTE, ["apo", "holo"]),
    "model":  (MODEL_PALETTE, ["alphafold3", "boltz", "chai1", "openfold3",
                               "protenix", "rosettafold3", "unknown"]),
}

# Default HDBSCAN search space for cluster_method='hdbscan', n_clusters='auto'
# (Optuna/TPE + DBCV tuning) -- same candidate values/naming as
# NPF_pocket_pipeline/notebook/msa_clustering/all_proteins_blosum62_pca_hdbscan.ipynb
# and AF3_NPF_pipeline/notebook/tm_conformation_clustering_*.ipynb, applied
# to this project's 2-D embedding. "cityblock" not "manhattan": same
# distance, but that name errors inside hdbscan.validity.validity_index.
HDBSCAN_MIN_SAMPLES_CANDIDATES = [3, 5, 10, 15, 20, 25, 30]
HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]
HDBSCAN_CLUSTER_SELECTION_METHODS = ["eom", "leaf"]
HDBSCAN_METRICS = ["euclidean", "cityblock"]

# color_by name -> (meta column, colorscale, (cmin, cmax) or None for data-range)
CONTINUOUS_COLOR_CONFIG = {
    "ptm":     ("ptm",     "Viridis", (0.0, 1.0)),
    "iptm":    ("iptm",    "Viridis", (0.0, 1.0)),  # NaN on AF3 apoform frames (no interface to score); 0.0 for the same case on the other 5 backends
    "seed":    ("seed",    "Turbo",   None),
    "rmsd_tm": ("rmsd_tm", "Plasma",  None),  # per-frame RMSD (A) to this ensemble's converged TM-helix mean
}

HOVER_COLS = ["unique_frame_id", "status", "model", "seed", "sample_index", "ptm", "iptm"]
HOVER_TEMPLATE_BODY = (
    "%{customdata[0]}<br>"
    "status: %{customdata[1]}  ·  model: %{customdata[2]}<br>"
    "seed %{customdata[3]}  ·  sample %{customdata[4]}<br>"
    "pTM: %{customdata[5]:.3f}  ·  ipTM: %{customdata[6]:.3f}<br>"
)


def _save_fig(fig, protein, filename):
    """Write a static PNG copy of fig under results/figures/<protein>/ (via
    kaleido) so plots survive a `results/` -> `results_vN/` rename instead
    of only living in the notebook's cell output / plotly's interactive
    fig.show()."""
    out_dir = FIG_ROOT / protein
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename
    fig.write_image(str(path), scale=2)
    return path


def _load_run(run_name):
    """Load one apo/holo run's aligned TM-Ca ensemble + per-frame metadata
    (model/backend, seed, sample index, pTM, RMSD-to-mean), written by
    scripts/tm_helix_alignment.py -- already pooled across every one of
    ABCfold's 6 backends (AlphaFold3, Boltz-2, Chai-1, OpenFold3, Protenix,
    RosettaFold3) x seed x diffusion/sample for this run, unlike
    AF3_NPF_pipeline's equivalent (AF3 only). `run_name` is a full
    apo/holo run identifier (e.g. 'NPF2.12_Q9LFX9__apo'), matching a
    results/tm_alignment/<run_name>/ directory."""
    npy = ALIGN_ROOT / run_name / "aligned_ca_tm.npy"
    csv = ALIGN_ROOT / run_name / "meta.csv"
    if not npy.exists():
        raise FileNotFoundError(
            f"{npy} not found -- run worflows/postprocessing/Snakefile "
            f"(scripts/tm_helix_alignment.py) for {run_name} first")
    coords = np.load(npy)                          # (n_frames, n_ca_tm, 3)
    meta   = pd.read_csv(csv)
    X      = coords.reshape(coords.shape[0], -1)   # flatten to (n_frames, n_ca_tm*3)
    return X, meta


def _kabsch_fit(P, Q):
    """Rotation R (3,3) and translation t (3,) such that (R @ P.T).T + t ~= Q.
    Same as kabsch() in scripts/tm_helix_alignment.py."""
    p_mean, q_mean = P.mean(axis=0), Q.mean(axis=0)
    Pc, Qc = P - p_mean, Q - q_mean
    U, _, Vt = np.linalg.svd(Pc.T @ Qc)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    t = q_mean - R @ p_mean
    return R, t


def _load_protein(protein):
    """Load the aligned, multi-backend TM-Ca ensemble for one BASE protein
    (e.g. 'NPF2.12_Q9LFX9'), merging its apoform and holoform ABCfold runs
    into a single pooled ensemble for analysis. Apoform always exists (or
    is expected to); holoform only when worflows/preprocessing/Snakefile's
    ligand_for() assigned a ligand AND that run has completed -- if
    there's no holoform run yet, only the apoform ensemble is returned.

    scripts/tm_helix_alignment.py's align_ensemble() converges each run
    (pooling all 6 backends already) to ITS OWN ensemble mean,
    independently -- apo and holo are separate ABCfold jobs with no shared
    global orientation, so their two reference frames can differ by an
    arbitrary rigid-body rotation/translation. Pooling them naively would
    make PCA/UMAP pick up that arbitrary offset instead of real
    ligand-induced conformational shifts, so holo's mean TM structure is
    Kabsch-refit onto apo's mean TM structure here (apo is the anchor
    since it always exists) and that one rigid-body transform is applied
    to every holo frame before pooling -- this only corrects a
    whole-ensemble offset, not per-frame noise, so a single fit is correct
    and sufficient.

    Adds three columns to meta: 'status' ('apo'/'holo'), 'source_run' (the
    underlying results/tm_alignment/<source_run>/ and
    results/abcfold/<source_run>/ directory name, needed by _reannotate to
    find the right CIFs), and 'unique_frame_id' (frame_id prefixed with
    status) since apo and holo runs independently repeat the same
    model/seed/sample numbering and would otherwise collide once pooled.
    'model' (the backend: alphafold3/boltz/chai1/openfold3/protenix/
    rosettafold3) is already a meta.csv column written by
    scripts/tm_helix_alignment.py -- untouched here, just carried through.
    """
    X_parts, meta_parts = [], []
    apo_mean = None
    for status in ("apo", "holo"):
        run_name = f"{protein}__{status}"
        if not (ALIGN_ROOT / run_name).exists():
            if status == "apo":
                raise FileNotFoundError(
                    f"{ALIGN_ROOT / run_name} not found -- apoform is expected "
                    f"for every protein; run worflows/postprocessing/Snakefile first")
            continue
        X, meta = _load_run(run_name)
        coords = X.reshape(X.shape[0], -1, 3)  # (n_frames, n_ca_tm, 3)

        if status == "apo":
            apo_mean = coords.mean(axis=0)
        else:
            R, t = _kabsch_fit(coords.mean(axis=0), apo_mean)
            flat = coords.reshape(-1, 3)
            coords = ((R @ flat.T).T + t).reshape(coords.shape)
            X = coords.reshape(coords.shape[0], -1)

        meta = meta.copy()
        meta["status"] = status
        meta["source_run"] = run_name
        meta["unique_frame_id"] = status + "_" + meta["frame_id"].astype(str)
        X_parts.append(X)
        meta_parts.append(meta)

    X    = np.concatenate(X_parts, axis=0)
    meta = pd.concat(meta_parts, ignore_index=True)
    return X, meta


# Mirrors scripts/tm_helix_alignment.py's BACKEND_PATTERNS / backend_of() /
# discover_predictions() / parse_frame_id() -- duplicated here (not
# imported) so this notebook stays self-contained, same convention as
# _kabsch_fit above mirroring that script's kabsch(). Keep in sync if the
# script's version changes. Only the frame_id half of parse_frame_id is
# needed here (to match a rediscovered CIF back to its meta.csv row by
# (source_run, frame_id) for reannotation) -- model/seed/sample_index are
# already columns tm_helix_alignment.py wrote to meta.csv itself.

BACKEND_PATTERNS = {
    "alphafold3":   "alphafold3",
    "boltz":        "boltz",
    "chai1":        "chai",
    "openfold3":    "openfold",
    "protenix":     "protenix",
    "rosettafold3": "rosettafold",
}


def _backend_of(path, predictions_dir):
    try:
        top = path.relative_to(predictions_dir).parts[0].lower()
    except (ValueError, IndexError):
        return "unknown"
    for backend, pattern in BACKEND_PATTERNS.items():
        if pattern in top:
            return backend
    return "unknown"


def _discover_abcfold_cifs(run_name):
    """Every model CIF ABCfold produced for one apo/holo run, pooled across
    all 6 backends x seed x diffusion/sample. Mirrors
    scripts/tm_helix_alignment.py's discover_predictions()."""
    predictions_dir = ABCFOLD_OUT_ROOT / run_name
    return sorted(c for c in predictions_dir.rglob("*.cif") if "templates" not in c.parts)


def _frame_id_for_cif(cif_path, predictions_dir):
    """Same frame_id derivation as scripts/tm_helix_alignment.py's
    parse_frame_id(), so a rediscovered CIF can be matched back to its
    meta.csv row by (source_run, frame_id)."""
    rel   = cif_path.relative_to(predictions_dir)
    model = _backend_of(cif_path, predictions_dir)
    m = re.search(r"seed-?(\d+)_sample-?(\d+)", str(rel), re.IGNORECASE)
    if m:
        return f"{model}_seed{m.group(1)}_sample{m.group(2)}"
    return f"{model}_{rel.with_suffix('')}".replace("/", "_")


def _ellipse_trace(mean, cov, color, n_std=1.5, n_pts=80):
    vals, vecs = np.linalg.eigh(cov)
    idx        = np.argsort(vals)[::-1]
    vals, vecs = vals[idx], vecs[:, idx]
    t          = np.linspace(0, 2 * np.pi, n_pts)
    pts        = n_std * (vecs * np.sqrt(np.maximum(vals, 0))) @ np.vstack(
                     [np.cos(t), np.sin(t)])
    x, y = mean[0] + pts[0], mean[1] + pts[1]
    return go.Scatter(
        x=np.append(x, x[0]), y=np.append(y, y[0]),
        mode="lines",
        line=dict(color=color, width=1.5, dash="dot"),
        showlegend=False, hoverinfo="skip",
    )


def _reannotate(protein, meta, labels, x_col, y_col, method_tag,
                max_per_cluster=20, sample_seed=42):
    """Symlink each structure's CIF into results/tm_reannotated/<protein>/<method_tag>/cluster_<k>/.

    `protein` is a BASE protein name; `meta` (from the merged
    _load_protein) covers both its apo and holo ABCfold runs, each under
    its own results/abcfold/<source_run>/ directory and pooling up to 6
    backends, so CIFs are looked up by (source_run, frame_id) rather than
    frame_id alone -- apo and holo runs independently repeat the same
    model/seed/sample numbering, so frame_id on its own is ambiguous once
    pooled. Looked up by frame_id (written into meta.csv by
    scripts/tm_helix_alignment.py) rather than positional zip, since that
    script may have skipped a frame mid-ensemble (Ca count mismatch) so a
    fresh CIF glob need not line up index-for-index with meta.csv.
    Symlinks are named "<unique_frame_id>.cif" (frame_id, itself already
    backend-prefixed by parse_frame_id, prefixed again with status) so
    apo/holo filenames never collide once multiple frames land in the same
    cluster_dir.

    Clusters routinely hold far more structures than is useful to load into
    ChimeraX at once, so at most `max_per_cluster` structures per cluster
    are randomly subsampled (without replacement, `sample_seed` for
    reproducibility) and only those get symlinked to disk. `assignments.csv`
    still lists every frame in the cluster (with a `symlinked` column) so
    the full membership stays available for downstream stats even though
    the on-disk CIF set is capped.

    Cluster ids are read from the data (`sorted(set(labels))`) rather than
    assumed to be `range(0, labels.max() + 1)`, so HDBSCAN's `-1` noise
    label gets its own `cluster_noise/` directory instead of being silently
    dropped (GMM labels are always a contiguous 0..k-1 range, so this is a
    no-op for the GMM path).
    """
    cif_by_key = {}
    for source_run in sorted(meta["source_run"].unique()):
        predictions_dir = ABCFOLD_OUT_ROOT / source_run
        for c in _discover_abcfold_cifs(source_run):
            cif_by_key[(source_run, _frame_id_for_cif(c, predictions_dir))] = c

    meta = meta.copy()
    meta["gmm_cluster"] = labels
    out_dir     = REANN_ROOT / protein / method_tag
    cluster_ids = sorted(set(int(l) for l in labels))

    assign_rows = []
    n_symlinked = 0
    for cid in cluster_ids:
        dir_name    = "cluster_noise" if cid == -1 else f"cluster_{cid}"
        cluster_dir = out_dir / dir_name
        cluster_dir.mkdir(parents=True, exist_ok=True)
        for stale in cluster_dir.iterdir():
            if stale.is_symlink():
                stale.unlink()

        cluster_rows = meta[meta["gmm_cluster"] == cid]
        sampled_idx = set(cluster_rows.sample(
            n=min(len(cluster_rows), max_per_cluster), random_state=sample_seed,
        ).index)

        for idx, row in cluster_rows.iterrows():
            cif = cif_by_key.get((row["source_run"], row["frame_id"]))
            symlinked = cif is not None and idx in sampled_idx
            if symlinked:
                dest = cluster_dir / f"{row['unique_frame_id']}.cif"
                # RosettaFold3 writes both a "_model.cif" and a
                # "_model_fixed.cif" per (seed, sample) -- near-identical
                # coordinates, but two distinct source CIFs that collide on
                # the same frame_id (parse_frame_id's regex only captures
                # seed/sample, not the "_fixed" suffix), so two meta.csv
                # rows can legitimately share one unique_frame_id. Skip
                # rather than crash on the second one; the first symlink
                # already represents this frame_id in this cluster_dir.
                if not dest.exists():
                    dest.symlink_to(cif.resolve())
                    n_symlinked += 1
            assign_rows.append({
                "protein":      protein,
                "status":       row["status"],
                "model":        row["model"],
                "seed":         row["seed"],
                "sample_index": row["sample_index"],
                "frame_id":     row["unique_frame_id"],
                "ptm":          row["ptm"],
                "iptm":         row["iptm"],
                "cluster":      cid,
                x_col:          round(float(row[x_col]), 4),
                y_col:          round(float(row[y_col]), 4),
                "symlinked":    symlinked,
            })
    pd.DataFrame(assign_rows).to_csv(out_dir / "assignments.csv", index=False)
    print(f"[reannotate] {protein}/{method_tag}: {n_symlinked} symlinks "
          f"(max {max_per_cluster}/cluster) of {len(assign_rows)} assignments -> {out_dir}")


def _fit_gmm_bic_sweep(xy, k_min=1, k_max=20, n_init=20, random_state=42):
    """Fit a GaussianMixture for every k in [k_min, k_max] on the 2-D embedding
    and return the model sitting at the knee of the BIC-vs-k curve.

    Mirrors find_best_k in NPF_pocket_pipeline/scripts/gmm_conformation.py:
    KneeLocator's default interp1d interpolation follows every point of the
    BIC curve exactly, so a single noisy value (e.g. a bad n_init restart)
    reads as a spurious knee right at the first bump. Fitting a polynomial
    through the curve first (interp_method="polynomial", degree capped
    relative to the number of k's swept) smooths that out and finds the
    real elbow instead. Falls back to the raw BIC minimum if KneeLocator
    finds no knee.
    """
    k_max = min(k_max, xy.shape[0] - 1)
    ks    = list(range(max(1, k_min), k_max + 1))

    gmms, bic_by_k = {}, {}
    for k in ks:
        gmm = GaussianMixture(n_components=k, covariance_type="full",
                               n_init=n_init, random_state=random_state)
        gmm.fit(xy)
        gmms[k]     = gmm
        bic_by_k[k] = float(gmm.bic(xy))

    best_k = ks[int(np.argmin([bic_by_k[k] for k in ks]))]
    if len(ks) >= 3:
        degree = min(7, max(1, len(ks) - 3))
        try:
            kl = KneeLocator(ks, [bic_by_k[k] for k in ks],
                              curve="convex", direction="decreasing",
                              interp_method="polynomial", polynomial_degree=degree)
            if kl.knee is not None:
                best_k = int(kl.knee)
        except Exception as e:
            print(f"[gmm-auto] WARNING: KneeLocator failed ({e}), falling back to BIC minimum")

    return gmms[best_k], best_k, bic_by_k


def _plot_bic_curve(protein, method_title, bic_by_k, best_k, method_tag):
    ks   = sorted(bic_by_k)
    bics = [bic_by_k[k] for k in ks]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=ks, y=bics, mode="lines+markers",
        line=dict(color="#1565C0", width=2), marker=dict(size=6),
        name="BIC",
    ))
    fig.add_trace(go.Scatter(
        x=[best_k], y=[bic_by_k[best_k]], mode="markers",
        marker=dict(size=14, color="#d62728", symbol="star"),
        name=f"knee k={best_k}",
    ))
    fig.update_layout(
        title=f"{protein}<br>{method_title}<br>BIC sweep k={ks[0]}-{ks[-1]}, knee k={best_k}",
        xaxis_title="n_components (k)", yaxis_title="BIC",
        template="plotly_white", height=380, width=520, showlegend=False,
    )
    _save_fig(fig, protein, f"{method_tag}_bic_k{best_k}.png")
    fig.show()


def _fit_hdbscan_dbcv_search(xy, min_samples_candidates=HDBSCAN_MIN_SAMPLES_CANDIDATES,
                              min_cluster_size_candidates=HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES,
                              cluster_selection_methods=HDBSCAN_CLUSTER_SELECTION_METHODS,
                              metrics=HDBSCAN_METRICS, n_trials=400, random_state=42):
    """Optuna/TPE search over (min_samples, min_cluster_size, cluster_selection_method,
    metric) for HDBSCAN on the 2-D embedding, scored by DBCV (Moulavi et al. 2014) via
    hdbscan.validity.validity_index -- same approach as
    select_hdbscan_hyperparams_and_cluster in
    NPF_pocket_pipeline/notebook/msa_clustering/all_proteins_blosum62_pca_hdbscan.ipynb,
    just applied to a 2-D embedding here instead of a full BLOSUM62-encoded
    sequence embedding. TPE models which regions of the search space tend to
    score well on DBCV as trials complete and concentrates later trials
    there, rather than sampling the grid uniformly at random. Combos
    yielding fewer than 2 clusters, or that error inside DBCV, score -1.0
    so they're never selected.
    """
    n = xy.shape[0]
    max_min_cluster_size = max(2, n // 5)
    candidate_min_cluster_sizes = [m for m in min_cluster_size_candidates if 2 <= m <= max_min_cluster_size]
    if not candidate_min_cluster_sizes:
        candidate_min_cluster_sizes = [max_min_cluster_size]

    grid_size = (len(min_samples_candidates) * len(candidate_min_cluster_sizes)
                 * len(cluster_selection_methods) * len(metrics))
    n_trials = min(n_trials, grid_size)

    def objective(trial):
        min_samples = trial.suggest_categorical("min_samples", list(min_samples_candidates))
        min_cluster_size = trial.suggest_categorical("min_cluster_size", candidate_min_cluster_sizes)
        cluster_selection_method = trial.suggest_categorical("cluster_selection_method", list(cluster_selection_methods))
        metric = trial.suggest_categorical("metric", list(metrics))
        try:
            labels = HDBSCAN(min_samples=min_samples, min_cluster_size=min_cluster_size,
                              cluster_selection_method=cluster_selection_method,
                              metric=metric, copy=False).fit(xy).labels_
            n_clust = len(set(c for c in labels if c >= 0))
            dbcv = float(validity_index(xy.astype(np.float64), labels, metric=metric)) if n_clust >= 2 else -1.0
        except Exception as e:
            print(f"[hdbscan-auto] combo ms={min_samples} mcs={min_cluster_size} "
                  f"{cluster_selection_method}/{metric} failed: {e}")
            labels, dbcv = None, -1.0
        trial.set_user_attr("labels", None if labels is None else labels.tolist())
        return dbcv

    sampler = optuna.samplers.TPESampler(seed=random_state)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_trial  = study.best_trial
    best_labels = best_trial.user_attrs["labels"]
    best = {
        "min_samples": best_trial.params["min_samples"],
        "min_cluster_size": best_trial.params["min_cluster_size"],
        "cluster_selection_method": best_trial.params["cluster_selection_method"],
        "metric": best_trial.params["metric"],
        "dbcv": best_trial.value,
    }
    labels = np.array(best_labels) if best_labels is not None else np.full(n, -1)
    return labels, best, study


def _plot_dbcv_search(protein, method_title, study, best, method_tag):
    dbcvs = sorted(study.trials_dataframe()["value"].fillna(-1.0).tolist(), reverse=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=list(range(len(dbcvs))), y=dbcvs, marker_color="#1565C0", name="DBCV"))
    fig.add_hline(y=best["dbcv"], line_dash="dash", line_color="#d62728",
                  annotation_text=f"best DBCV={best['dbcv']:.3f}")
    fig.update_layout(
        title=f"{protein}<br>{method_title}<br>HDBSCAN Optuna/TPE search, {len(dbcvs)} trials",
        xaxis_title="trial (sorted by DBCV)", yaxis_title="DBCV",
        template="plotly_white", height=380, width=520, showlegend=False,
    )
    _save_fig(fig, protein, f"{method_tag}_hdbscan_dbcv_search.png")
    fig.show()


def _plot_embedding(protein, meta, xy, x_col, y_col, method_tag, method_title,
                     color_by="model", cluster_method="gmm", n_clusters=None,
                     max_per_cluster=20, auto_k_min=1, auto_k_max=20,
                     hdbscan_min_cluster_size=None, hdbscan_min_samples=None,
                     hdbscan_cluster_selection_method="eom", hdbscan_metric="euclidean",
                     hdbscan_n_trials=40,
                     marker_size=6, opacity=0.7, axis_titles=("dim 1", "dim 2")):
    """Shared scatter / cluster / reannotate renderer for PCA, t-SNE and UMAP.

    color_by is a CATEGORICAL_COLOR_CONFIG key ("model" default -- which of
    the 6 ABCfold backends produced each frame, see MODEL_PALETTE; or
    "status" -- apo/holo) or a CONTINUOUS_COLOR_CONFIG key ("ptm", "iptm", "seed",
    or "rmsd_tm"). n_clusters, if set, fits cluster_method ("gmm" default,
    or "hdbscan") on the 2-D embedding instead and colours by cluster:

    - cluster_method="gmm": an int n_clusters fits exactly that many GMM
      components (manual); "auto" sweeps auto_k_min..auto_k_max components
      and picks the BIC-curve knee instead (see _fit_gmm_bic_sweep).
    - cluster_method="hdbscan": n_clusters="auto" searches
      (hdbscan_min_cluster_size, hdbscan_min_samples,
      hdbscan_cluster_selection_method, hdbscan_metric) with Optuna/TPE
      scored by DBCV (see _fit_hdbscan_dbcv_search, hdbscan_n_trials
      trials); "manual" fits HDBSCAN directly with the explicit hdbscan_*
      arguments (hdbscan_min_cluster_size is required in that case). Points
      HDBSCAN calls noise (-1) are shown as unclustered ("x" markers)
      rather than being assigned a colour.

    max_per_cluster caps how many of each cluster's CIFs get symlinked for
    reannotation (see _reannotate); HDBSCAN's noise points get their own
    cluster_noise/ subsample rather than being dropped.

    Every figure this function produces (the BIC/DBCV diagnostic plot, when
    applicable, and the main embedding scatter) is also written as a static
    PNG under results/figures/<protein>/ via _save_fig, tagged with the same
    method/cluster identifier used for reannotation symlinks -- so plots
    survive a `results/` -> `results_vN/` rename instead of only existing as
    notebook cell output.
    """
    meta = meta.copy()
    meta[x_col], meta[y_col] = xy[:, 0], xy[:, 1]

    fig = go.Figure()

    if n_clusters is not None and cluster_method == "gmm":
        # -- GMM clustering on the 2-D embedding (manual k or auto BIC-knee) --
        if n_clusters == "auto":
            gmm, k_used, bic_by_k = _fit_gmm_bic_sweep(xy, k_min=auto_k_min, k_max=auto_k_max)
            _plot_bic_curve(protein, method_title, bic_by_k, k_used, method_tag)
            cluster_label = f"GMM auto (BIC knee) k={k_used}"
        else:
            k_used = n_clusters
            gmm = GaussianMixture(n_components=k_used, covariance_type="full",
                                   n_init=20, random_state=42)
            gmm.fit(xy)
            cluster_label = f"GMM k={k_used}"

        labels = gmm.predict(xy)
        meta["gmm_cluster"] = labels
        fig_tag = f"{method_tag}_k{k_used}"

        cdata = meta[HOVER_COLS].fillna("?").values
        for k in range(k_used):
            sub   = meta[meta["gmm_cluster"] == k]
            color = GMM_PALETTE[k % len(GMM_PALETTE)]
            fig.add_trace(go.Scatter(
                x=sub[x_col], y=sub[y_col],
                mode="markers",
                marker=dict(size=marker_size, color=color, opacity=opacity),
                name=f"cluster {k}",
                customdata=cdata[meta["gmm_cluster"].values == k],
                hovertemplate=(
                    HOVER_TEMPLATE_BODY +
                    f"{x_col}: %{{x:.3f}}  {y_col}: %{{y:.3f}}"
                    "<extra></extra>"
                ),
            ))
            fig.add_trace(_ellipse_trace(gmm.means_[k], gmm.covariances_[k], color))

        legend_title = "GMM cluster"
        _reannotate(protein, meta, labels, x_col, y_col, fig_tag,
                    max_per_cluster=max_per_cluster)

    elif n_clusters is not None and cluster_method == "hdbscan":
        # -- HDBSCAN clustering on the 2-D embedding (auto DBCV-tuned or manual) --
        if n_clusters == "auto":
            labels, best, study = _fit_hdbscan_dbcv_search(xy, n_trials=hdbscan_n_trials)
            _plot_dbcv_search(protein, method_title, study, best, method_tag)
            cluster_label = (f"HDBSCAN auto (DBCV={best['dbcv']:.3f}) "
                              f"mcs={best['min_cluster_size']} ms={best['min_samples']} "
                              f"{best['cluster_selection_method']}/{best['metric']}")
            fig_tag = f"{method_tag}_hdbscan_auto"
        elif n_clusters == "manual":
            if hdbscan_min_cluster_size is None:
                raise ValueError(
                    "cluster_method='hdbscan' with n_clusters='manual' requires "
                    "hdbscan_min_cluster_size to be set")
            labels = HDBSCAN(min_cluster_size=hdbscan_min_cluster_size,
                              min_samples=hdbscan_min_samples,
                              cluster_selection_method=hdbscan_cluster_selection_method,
                              metric=hdbscan_metric).fit(xy).labels_
            cluster_label = (f"HDBSCAN manual mcs={hdbscan_min_cluster_size} "
                              f"ms={hdbscan_min_samples} "
                              f"{hdbscan_cluster_selection_method}/{hdbscan_metric}")
            fig_tag = f"{method_tag}_hdbscan_manual_mcs{hdbscan_min_cluster_size}"
        else:
            raise ValueError(
                "cluster_method='hdbscan' requires n_clusters='auto' or 'manual' "
                f"(got {n_clusters!r})")

        meta["gmm_cluster"] = labels
        cdata = meta[HOVER_COLS].fillna("?").values
        cluster_ids = sorted(c for c in set(labels) if c >= 0)
        for i, k in enumerate(cluster_ids):
            sub   = meta[meta["gmm_cluster"] == k]
            color = GMM_PALETTE[i % len(GMM_PALETTE)]
            fig.add_trace(go.Scatter(
                x=sub[x_col], y=sub[y_col],
                mode="markers",
                marker=dict(size=marker_size, color=color, opacity=opacity),
                name=f"cluster {k}",
                customdata=cdata[meta["gmm_cluster"].values == k],
                hovertemplate=(
                    HOVER_TEMPLATE_BODY +
                    f"{x_col}: %{{x:.3f}}  {y_col}: %{{y:.3f}}"
                    "<extra></extra>"
                ),
            ))

        noise = meta[meta["gmm_cluster"] == -1]
        if not noise.empty:
            fig.add_trace(go.Scatter(
                x=noise[x_col], y=noise[y_col],
                mode="markers",
                marker=dict(size=marker_size - 1, color="#aaa", opacity=0.4, symbol="x"),
                name="noise (HDBSCAN)",
                customdata=noise[HOVER_COLS].fillna("?").values,
                hovertemplate=(
                    HOVER_TEMPLATE_BODY +
                    f"{x_col}: %{{x:.3f}}  {y_col}: %{{y:.3f}}"
                    "<extra></extra>"
                ),
            ))

        legend_title = "HDBSCAN cluster"
        _reannotate(protein, meta, labels, x_col, y_col, fig_tag,
                    max_per_cluster=max_per_cluster)

    elif color_by in CATEGORICAL_COLOR_CONFIG:
        # -- categorical colour scale: "model" (default, 6 ABCfold backends) or "status" --
        palette, order = CATEGORICAL_COLOR_CONFIG[color_by]
        present = set(meta[color_by].dropna())
        for category in [c for c in order if c in present]:
            sub = meta[meta[color_by] == category]
            if sub.empty:
                continue
            fig.add_trace(go.Scatter(
                x=sub[x_col], y=sub[y_col],
                mode="markers",
                marker=dict(size=marker_size, color=palette.get(category, "#7f7f7f"), opacity=opacity),
                name=category,
                customdata=sub[HOVER_COLS].fillna("?").values,
                hovertemplate=(
                    HOVER_TEMPLATE_BODY +
                    f"{x_col}: %{{x:.3f}}  {y_col}: %{{y:.3f}}"
                    "<extra></extra>"
                ),
            ))

        cluster_label = f"color_by={color_by}"
        legend_title  = color_by
        fig_tag = f"{method_tag}_color_{color_by}"

    else:
        # -- continuous colour scale (pTM, seed, or rmsd_tm) --
        if color_by not in CONTINUOUS_COLOR_CONFIG:
            raise ValueError(f"Unknown color_by {color_by!r} -- expected one of "
                              f"{list(CATEGORICAL_COLOR_CONFIG)} or {list(CONTINUOUS_COLOR_CONFIG)}")
        col, colorscale, bounds = CONTINUOUS_COLOR_CONFIG[color_by]
        has_val = meta[col].notna()
        if not has_val.any():
            raise ValueError(
                f"color_by={color_by!r} has no non-NaN values anywhere in this ensemble "
                f"(e.g. find_confidence() in scripts/tm_helix_alignment.py found no pTM "
                f"for any backend here) -- try color_by='model' or 'status' instead")
        sub, missing = meta[has_val], meta[~has_val]
        cmin, cmax = bounds if bounds is not None else (sub[col].min(), sub[col].max())
        fig.add_trace(go.Scatter(
            x=sub[x_col], y=sub[y_col],
            mode="markers",
            marker=dict(size=marker_size, color=sub[col], colorscale=colorscale,
                        cmin=cmin, cmax=cmax, opacity=opacity,
                        colorbar=dict(title=color_by)),
            customdata=sub[HOVER_COLS].fillna("?").values,
            hovertemplate=(
                HOVER_TEMPLATE_BODY +
                f"{x_col}: %{{x:.3f}}  {y_col}: %{{y:.3f}}"
                "<extra></extra>"
            ),
            showlegend=False,
        ))
        if not missing.empty:
            fig.add_trace(go.Scatter(
                x=missing[x_col], y=missing[y_col],
                mode="markers",
                marker=dict(size=marker_size - 1, color="#aaa",
                            opacity=0.4, symbol="x"),
                name=f"{color_by} missing",
            ))

        cluster_label = f"color_by={color_by}"
        legend_title  = color_by
        fig_tag = f"{method_tag}_color_{color_by}"

    fig.update_layout(
        title=f"{protein}<br>{method_title}<br>{cluster_label}",
        xaxis_title=axis_titles[0], yaxis_title=axis_titles[1],
        legend_title=legend_title,
        template="plotly_white",
        height=540, width=680,
        hovermode="closest",
        hoverlabel=dict(font_size=11, namelength=0),
        hoverdistance=30,
    )
    _save_fig(fig, protein, f"{fig_tag}_embedding.png")
    fig.show()


def _plot_pca_1d_fallback(protein, meta, pc1, evr1, method_tag, x_col="pc_x",
                           max_per_cluster=20, auto_k_min=1, auto_k_max=20):
    """Histogram + 1-D GMM auto (BIC-knee) clustering fallback for plot_pca
    when PC1 alone already explains >95% of the variance (see fallback_1d
    on plot_pca) -- at that point PC2 is close to pure noise, so the usual
    2-D scatter is misleading; a 1-D histogram of PC1, split by GMM
    cluster, is the honest representation instead. Always GMM auto
    (BIC-knee sweep, see _fit_gmm_bic_sweep) regardless of plot_pca's own
    cluster_method/n_components arguments -- HDBSCAN's tuning knobs don't
    carry over to a single dimension.
    """
    meta = meta.copy()
    meta[x_col] = pc1

    gmm, k_used, bic_by_k = _fit_gmm_bic_sweep(pc1.reshape(-1, 1), k_min=auto_k_min, k_max=auto_k_max)
    method_title = f"PCA  PC1-only fallback ({evr1:.1%} var > 95%)"
    _plot_bic_curve(protein, method_title, bic_by_k, k_used, method_tag)

    labels = gmm.predict(pc1.reshape(-1, 1))
    meta["gmm_cluster"] = labels
    fig_tag = f"{method_tag}_k{k_used}_hist1d"

    fig = go.Figure()
    for k in range(k_used):
        sub   = meta[meta["gmm_cluster"] == k]
        color = GMM_PALETTE[k % len(GMM_PALETTE)]
        fig.add_trace(go.Histogram(
            x=sub[x_col], name=f"cluster {k}", marker_color=color, opacity=0.7,
        ))
    fig.update_layout(
        title=f"{protein}<br>{method_title}<br>GMM auto (BIC knee) k={k_used}",
        xaxis_title=f"PC1  ({evr1:.1%} var)", yaxis_title="count",
        barmode="overlay", legend_title="GMM cluster",
        template="plotly_white", height=430, width=680,
    )
    _save_fig(fig, protein, f"{fig_tag}_hist.png")
    fig.show()

    _reannotate(protein, meta, labels, x_col=x_col, y_col=x_col, method_tag=fig_tag,
                max_per_cluster=max_per_cluster)


def plot_pca(protein: str, color_by: str = "model", cluster_method: str = "gmm", n_components=None,
             max_per_cluster: int = 20, auto_k_min: int = 1, auto_k_max: int = 20,
             hdbscan_min_cluster_size=None, hdbscan_min_samples=None,
             hdbscan_cluster_selection_method: str = "eom", hdbscan_metric: str = "euclidean",
             hdbscan_n_trials: int = 40,
             pc_x: int = 1, pc_y: int = 2,
             marker_size: int = 6, opacity: float = 0.7,
             fallback_1d: bool = True):
    """PCA scatter of the aligned, multi-backend TM-Ca ensemble for one BASE
    protein, apo and holo runs pooled together (see _load_protein).

    Parameters
    ----------
    protein       BASE protein identifier, e.g. "NPF2.12_Q9LFX9" -- its
                  apoform and holoform ABCfold runs are merged into one
                  ensemble before plotting (see _load_protein), each run
                  itself already pooling up to 6 backends x every seed x
                  diffusion/sample (scripts/tm_helix_alignment.py).
    color_by      "model" (default -- categorical, which of the 6 ABCfold
                  backends produced each frame, see MODEL_PALETTE -- the
                  key new axis this pipeline exists for, since a single
                  backend's diffusion doesn't always recover every
                  conformation a second one finds), "status" (categorical
                  apo/holo, see STATUS_PALETTE), "ptm"/"iptm" (continuous 0-1
                  predicted TM-score confidence), "seed" (continuous,
                  data-range colour scale -- spot outlier seeds), or
                  "rmsd_tm" (continuous, per-frame RMSD in A to this
                  ensemble's converged TM-helix mean). Ignored when
                  n_components is set, or when the fallback_1d histogram
                  fires.
    cluster_method  "gmm" (default) or "hdbscan" -- which algorithm n_components
                  fits on the 2-D PCA coordinates. Ignored when the
                  fallback_1d histogram fires (that path is always GMM auto).
    n_components  clustering mode, colouring by cluster (CIFs symlinked into
                  results/tm_reannotated):
                    - cluster_method="gmm": int fits exactly that many
                      components (manual); 'auto' sweeps auto_k_min..auto_k_max
                      and picks the knee of the BIC-vs-k curve (kneed,
                      polynomial smoothing -- see _fit_gmm_bic_sweep), with a
                      BIC diagnostic plot alongside the embedding. Ellipses
                      are drawn from the GMM covariances.
                    - cluster_method="hdbscan": 'auto' searches
                      (hdbscan_min_cluster_size, hdbscan_min_samples,
                      hdbscan_cluster_selection_method, hdbscan_metric) with
                      Optuna/TPE scored by DBCV (hdbscan_n_trials trials --
                      see _fit_hdbscan_dbcv_search), with a DBCV diagnostic
                      plot alongside the embedding; 'manual' fits HDBSCAN
                      directly with the explicit hdbscan_* arguments below
                      (hdbscan_min_cluster_size is then required). Points
                      HDBSCAN calls noise (-1) are shown unclustered.
    max_per_cluster  cap on how many CIFs per cluster get symlinked for
                  reannotation (randomly subsampled, including HDBSCAN's
                  noise cluster). Only used when n_components is set, or
                  when the fallback_1d histogram fires.
    auto_k_min, auto_k_max  BIC sweep range for cluster_method="gmm",
                  n_components='auto' -- also used by the fallback_1d
                  histogram's GMM-auto clustering.
    hdbscan_min_cluster_size, hdbscan_min_samples, hdbscan_cluster_selection_method,
    hdbscan_metric  explicit HDBSCAN hyperparameters for
                  cluster_method="hdbscan", n_components='manual'.
    hdbscan_n_trials  Optuna trial budget for cluster_method="hdbscan",
                  n_components='auto'.
    pc_x, pc_y    which PCs to plot (1-indexed). Irrelevant when the
                  fallback_1d histogram fires (PC1 only, by construction).
    fallback_1d   when True (default) and PC1 alone already explains >95%
                  of the variance, PC2 carries almost no real signal, so
                  the usual 2-D scatter is misleading -- plot_pca switches
                  to a 1-D histogram of PC1 split by GMM-auto (BIC-knee)
                  clusters instead (see _plot_pca_1d_fallback), ignoring
                  color_by/cluster_method/n_components/pc_x/pc_y entirely.
                  Set False to always force the normal 2-D plot.
    """
    X, meta = _load_protein(protein)
    n_pc    = min(max(pc_x, pc_y, 5), X.shape[1])
    pca     = PCA(n_components=n_pc)
    coords  = pca.fit_transform(X)
    evr     = pca.explained_variance_ratio_

    if fallback_1d and evr[0] > 0.95:
        _plot_pca_1d_fallback(
            protein, meta, coords[:, 0], evr[0], method_tag="pca",
            max_per_cluster=max_per_cluster, auto_k_min=auto_k_min, auto_k_max=auto_k_max,
        )
        return

    xy      = coords[:, [pc_x - 1, pc_y - 1]]

    _plot_embedding(
        protein, meta, xy, x_col="pc_x", y_col="pc_y", method_tag="pca",
        method_title=f"PCA  PC{pc_x} vs PC{pc_y}  TM-Ca ({X.shape[1] // 3} atoms)",
        color_by=color_by, cluster_method=cluster_method, n_clusters=n_components,
        max_per_cluster=max_per_cluster, auto_k_min=auto_k_min, auto_k_max=auto_k_max,
        hdbscan_min_cluster_size=hdbscan_min_cluster_size, hdbscan_min_samples=hdbscan_min_samples,
        hdbscan_cluster_selection_method=hdbscan_cluster_selection_method,
        hdbscan_metric=hdbscan_metric, hdbscan_n_trials=hdbscan_n_trials,
        marker_size=marker_size, opacity=opacity,
        axis_titles=(f"PC{pc_x}  ({evr[pc_x - 1]:.1%} var)",
                     f"PC{pc_y}  ({evr[pc_y - 1]:.1%} var)"),
    )


def plot_tsne(protein: str, color_by: str = "model", cluster_method: str = "gmm", n_clusters=None,
              max_per_cluster: int = 20, auto_k_min: int = 1, auto_k_max: int = 20,
              hdbscan_min_cluster_size=None, hdbscan_min_samples=None,
              hdbscan_cluster_selection_method: str = "eom", hdbscan_metric: str = "euclidean",
              hdbscan_n_trials: int = 40,
              perplexity: float = 30.0, random_state: int = 42,
              marker_size: int = 6, opacity: float = 0.7):
    """t-SNE scatter of the aligned, multi-backend TM-Ca ensemble for one
    BASE protein, apo and holo runs pooled together (see _load_protein).

    Parameters mirror plot_pca / plot_umap: color_by defaults to "model"
    (categorical, which of the 6 ABCfold backends produced each frame);
    cluster_method selects "gmm" (default: n_clusters=int manual or 'auto'
    BIC-knee sweep) or "hdbscan" (n_clusters='auto' Optuna/DBCV search or
    'manual' with explicit hdbscan_* arguments) on the 2-D t-SNE embedding
    (max_per_cluster caps how many CIFs per cluster get symlinked for
    reannotation). perplexity is capped automatically for small ensembles.
    """
    X, meta = _load_protein(protein)
    perplexity = min(perplexity, max(5.0, (len(meta) - 1) / 3))
    tsne = TSNE(n_components=2, perplexity=perplexity,
                random_state=random_state, init="pca")
    xy = tsne.fit_transform(X)

    _plot_embedding(
        protein, meta, xy, x_col="tsne_x", y_col="tsne_y", method_tag="tsne",
        method_title=f"t-SNE  TM-Ca ({X.shape[1] // 3} atoms)  perplexity={perplexity:.0f}",
        color_by=color_by, cluster_method=cluster_method, n_clusters=n_clusters,
        max_per_cluster=max_per_cluster, auto_k_min=auto_k_min, auto_k_max=auto_k_max,
        hdbscan_min_cluster_size=hdbscan_min_cluster_size, hdbscan_min_samples=hdbscan_min_samples,
        hdbscan_cluster_selection_method=hdbscan_cluster_selection_method,
        hdbscan_metric=hdbscan_metric, hdbscan_n_trials=hdbscan_n_trials,
        marker_size=marker_size, opacity=opacity,
        axis_titles=("t-SNE-1", "t-SNE-2"),
    )


def plot_umap(protein: str, color_by: str = "model", cluster_method: str = "gmm", n_clusters=None,
              max_per_cluster: int = 20, auto_k_min: int = 1, auto_k_max: int = 20,
              hdbscan_min_cluster_size=None, hdbscan_min_samples=None,
              hdbscan_cluster_selection_method: str = "eom", hdbscan_metric: str = "euclidean",
              hdbscan_n_trials: int = 40,
              n_neighbors: int = 15, min_dist: float = 0.1,
              metric: str = "euclidean", random_state: int = 42,
              marker_size: int = 6, opacity: float = 0.7):
    """UMAP scatter of the aligned, multi-backend TM-Ca ensemble for one
    BASE protein, apo and holo runs pooled together (see _load_protein).

    Parameters mirror plot_pca / plot_tsne: color_by defaults to "model"
    (categorical, which of the 6 ABCfold backends produced each frame);
    "status" (apo/holo), "ptm", "iptm", "seed" and "rmsd_tm" are the other options.
    cluster_method selects which algorithm clusters the 2-D UMAP embedding
    instead:

    - "gmm" (default): n_clusters=int fits exactly that many components
      (manual), or 'auto' sweeps auto_k_min..auto_k_max and picks the
      BIC-vs-k curve's knee (kneed, polynomial-smoothed -- see
      _fit_gmm_bic_sweep -- a BIC diagnostic plot is shown alongside the
      embedding).
    - "hdbscan": n_clusters='auto' searches (hdbscan_min_cluster_size,
      hdbscan_min_samples, hdbscan_cluster_selection_method, hdbscan_metric)
      with Optuna/TPE scored by DBCV (hdbscan_n_trials trials -- see
      _fit_hdbscan_dbcv_search, a DBCV diagnostic plot is shown alongside
      the embedding), or n_clusters='manual' fits HDBSCAN directly with the
      explicit hdbscan_* arguments (hdbscan_min_cluster_size is then
      required). Note: `metric` here is UMAP's own embedding metric, not
      HDBSCAN's -- that's `hdbscan_metric`.

    max_per_cluster caps how many CIFs per cluster get symlinked for
    reannotation (randomly subsampled, including HDBSCAN's noise cluster).
    """
    X, meta = _load_protein(protein)
    reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist,
                         metric=metric, random_state=random_state)
    xy = reducer.fit_transform(X)

    _plot_embedding(
        protein, meta, xy, x_col="umap_x", y_col="umap_y", method_tag="umap",
        method_title=(f"UMAP  TM-Ca ({X.shape[1] // 3} atoms)  "
                      f"(n_neighbors={n_neighbors}, min_dist={min_dist})"),
        color_by=color_by, cluster_method=cluster_method, n_clusters=n_clusters,
        max_per_cluster=max_per_cluster, auto_k_min=auto_k_min, auto_k_max=auto_k_max,
        hdbscan_min_cluster_size=hdbscan_min_cluster_size, hdbscan_min_samples=hdbscan_min_samples,
        hdbscan_cluster_selection_method=hdbscan_cluster_selection_method,
        hdbscan_metric=hdbscan_metric, hdbscan_n_trials=hdbscan_n_trials,
        marker_size=marker_size, opacity=opacity,
        axis_titles=("UMAP-1", "UMAP-2"),
    )


print("Setup done.  plot_pca(protein) | plot_tsne(protein) | plot_umap(protein)  "
      "(protein = BASE name, apo+holo pooled, each already pooling all 6 ABCfold backends)"
      "  (default color_by='model' [alphafold3/boltz/chai1/openfold3/protenix/rosettafold3], "
      "or 'status' [apo/holo] / 'ptm' / 'iptm' / 'seed' / 'rmsd_tm'; cluster_method='gmm'; "
      "pass n_clusters=k [manual] / n_clusters='auto' [BIC-knee sweep] for GMM clustering, or "
      "cluster_method='hdbscan' with n_clusters='auto' [Optuna/DBCV search] / 'manual' "
      "for HDBSCAN clustering)")


Setup done.  plot_pca(protein) | plot_tsne(protein) | plot_umap(protein)  (protein = BASE name, apo+holo pooled, each already pooling all 6 ABCfold backends)  (default color_by='model' [alphafold3/boltz/chai1/openfold3/protenix/rosettafold3], or 'status' [apo/holo] / 'ptm' / 'seed' / 'rmsd_tm'; cluster_method='gmm'; pass n_clusters=k [manual] / n_clusters='auto' [BIC-knee sweep] for GMM clustering, or cluster_method='hdbscan' with n_clusters='auto' [Optuna/DBCV search] / 'manual' for HDBSCAN clustering)


## Ligand / protein metadata

`ligand_for()` mirrors `worflows/preprocessing/Snakefile`'s function of the
same name (that file is a Snakefile -- uses `checkpoint`/`rule`/
`configfile` directives -- not a plain importable module, so the
protein->ligand lists are mirrored here rather than imported; keep the two
in sync if `ligand_for()` changes). `category_of()` buckets its result into
`"gibberellin"` (GA1), `"nitrate"`, `"other_ligand"` (every other assigned
ligand) or `"apoform"` (no ligand assigned at all) -- used below only to
annotate each protein's markdown header, not to split this notebook into
several files (unlike `AF3_NPF_pipeline`, see the title cell above).
`PROTEINS` is every protein discovered under `results/tm_alignment/`, no
filtering. Ligand SMILES are read directly from `config.yaml`'s `ligands:`
section.


In [2]:
import yaml

_config = yaml.safe_load((ROOT / "config.yaml").read_text())
LIGANDS = _config["ligands"]  # ligand key -> {"smiles": ...}, from config.yaml

# Mirrors worflows/preprocessing/Snakefile's protein->ligand assignment lists.
HC_IMPORTERS = ['NPF3.1', 'NPF4.1', 'NPF2.12', 'NPF2.13', 'NPF2.10', 'NPF2.5']
NITRATE_TRANSPORTERS = ['NPF1.1', 'NPF1.2', 'NPF1.3', 'NPF2.3', 'NPF2.4', 'NPF2.7', 'NPF2.9', 'NPF2.11', 'NPF4.6', 'NPF5.5', 'NPF5.8', 'NPF5.9', 'NPF5.10', 'NPF5.11', 'NPF5.12', 'NPF5.14', 'NPF5.16', 'NPF6.2', 'NPF6.3', 'NPF7.2', 'NPF7.3', 'NPF8.5']
ABA_TRANSPORTERS = ['NPF2.14', 'NPF4.2', 'NPF4.5', 'NPF4.7', 'NPF5.1', 'NPF5.2', 'NPF5.3', 'NPF5.7']
AUXIN_TRANSPORTERS = ['NPF7.1']
GLYCERATE_TRANSPORTERS = ['NPF8.4']
DIMETHYLARSENATE_TRANSPORTERS = ['NPF8.1', 'NPF8.2']
JA_ILE_TRANSPORTERS = ['NPF2.6']
DIPEPTIDE_TRANSPORTERS = ['NPF8.3']
FLAVONOID_TRANSPORTERS = ['NPF2.8']
POLYAMINE_TRANSPORTERS = ['NPF6.4']
LOW_CONFIDENCE_GA_IMPORTERS = ['NPF2.1', 'NPF5.6']


def ligand_for(npf_name):
    """Same precedence as worflows/preprocessing/Snakefile's ligand_for();
    npf_name is the fasta basename, e.g. 'NPF2.12' (not the full
    '<npf_name>_<uniprot>' protein directory name)."""
    if npf_name in HC_IMPORTERS:
        return "GA1"
    if npf_name in NITRATE_TRANSPORTERS:
        return "nitrate"
    if npf_name in ABA_TRANSPORTERS:
        return "ABA"
    if npf_name in AUXIN_TRANSPORTERS:
        return "auxin"
    if npf_name in GLYCERATE_TRANSPORTERS:
        return "glycerate"
    if npf_name in DIMETHYLARSENATE_TRANSPORTERS:
        return "dimethylarsenate"
    if npf_name in DIPEPTIDE_TRANSPORTERS:
        return "glycylglycine"
    if npf_name in FLAVONOID_TRANSPORTERS:
        return "quercetin-3-O-sophoroside"
    if npf_name in POLYAMINE_TRANSPORTERS:
        return "spermidine"
    if npf_name in JA_ILE_TRANSPORTERS:
        return "JA-Ile"
    if npf_name in LOW_CONFIDENCE_GA_IMPORTERS:
        return "GA1"
    return None


def category_of(npf_name):
    """Buckets ligand_for()'s result for this protein's markdown header --
    this notebook itself is not split by category (see the title cell)."""
    key = ligand_for(npf_name)
    if key == "GA1":
        return "gibberellin"
    if key == "nitrate":
        return "nitrate"
    if key is None:
        return "apoform"
    return "other_ligand"


def _base_protein_name(dirname):
    """Strip the '__apo'/'__holo' ABCfold-run suffix, if present, to
    recover the base protein name plot_pca/plot_tsne/plot_umap expect --
    they pool both runs internally via _load_protein."""
    if dirname.endswith("__apo") or dirname.endswith("__holo"):
        return dirname.rsplit("__", 1)[0]
    return dirname


ALL_PROTEINS = sorted({
    _base_protein_name(p.name) for p in ALIGN_ROOT.iterdir()
    if p.is_dir() and not p.name.startswith(".")
})

# BASE protein (e.g. "NPF2.12_Q9LFX9") -> ligand key or None (apoform only)
PROTEIN_LIGAND = {
    protein: ligand_for(protein.rsplit("_", 1)[0]) for protein in ALL_PROTEINS
}

# This notebook's slice: every protein under results/tm_alignment/ (no
# category filter, unlike AF3_NPF_pipeline -- see title cell).
PROTEINS = ALL_PROTEINS

print(f"{len(PROTEINS)} protein(s) under results/tm_alignment/")
for protein in PROTEINS:
    key = PROTEIN_LIGAND[protein]
    if key is None:
        print(f"  {protein:20s} apoform only  [{category_of(protein.rsplit('_', 1)[0])}]")
    else:
        print(f"  {protein:20s} holoform ligand: {key}  ({LIGANDS[key]['smiles']})  [{category_of(protein.rsplit('_', 1)[0])}]")


1 protein(s) under results/tm_alignment/
  NPF2.12_Q9LFX9       holoform ligand: GA1  (C[C@]12[C@@H](O)CC[C@@]3(OC1=O)[C@@H]4CC[C@]5(O)C[C@]4(CC5=C)[C@H]([C@H]23)C(=O)O)  [gibberellin]


In [3]:
print(f"Backends discovered ({list(BACKEND_PATTERNS)}), per-protein frame counts from results/tm_alignment/:")
for protein in PROTEINS:
    counts = {}
    for status in ("apo", "holo"):
        csv_path = ALIGN_ROOT / f"{protein}__{status}" / "meta.csv"
        if not csv_path.exists():
            continue
        for model, n in pd.read_csv(csv_path)["model"].value_counts().items():
            counts[model] = counts.get(model, 0) + int(n)
    if counts:
        summary = ", ".join(f"{m}({counts[m]})" for m in BACKEND_PATTERNS if m in counts)
        print(f"  {protein:20s} {summary}")
    else:
        print(f"  {protein:20s} NO DATA")


Backends discovered (['alphafold3', 'boltz', 'chai1', 'openfold3', 'protenix', 'rosettafold3']), per-protein frame counts from results/tm_alignment/:
  NPF2.12_Q9LFX9       alphafold3(101), boltz(100), chai1(100), openfold3(4000), protenix(100), rosettafold3(220)


## Per-protein cells

One markdown + code cell pair per protein discovered under
`results/tm_alignment/` at the time this notebook was built. Each code
cell: a plain `color_by='model'` PCA first (which of the 6 backends
produced each point, no clustering), then an HDBSCAN/DBCV-tuned clustering
pass. Edit a cell's arguments directly (`color_by`, `cluster_method`,
manual `n_clusters`/`hdbscan_min_cluster_size`, etc.) for one-off
exploration on that protein without touching any other cell.

As more proteins' stage 6 (`scripts/tm_helix_alignment.py`) output lands
under `results/tm_alignment/`, re-run the metadata cell above to refresh
`PROTEINS`, then add a new markdown+code cell pair for it here.


### `NPF2.12_Q9LFX9` -- holoform ligand: **GA1** (Gibberellin A1) -- apoform-only data currently available

`C[C@]12[C@@H](O)CC[C@@]3(OC1=O)[C@@H]4CC[C@]5(O)C[C@]4(CC5=C)[C@H]([C@H]23)C(=O)O`

First protein x form this pipeline completed stage 6 for
(`results/abcfold/NPF2.12_Q9LFX9__apo`, `results/tm_alignment/NPF2.12_Q9LFX9__apo`).
Holoform hasn't been run yet, so `_load_protein` below only has the
apoform ensemble to pool -- `color_by='status'` would show a single
category; `color_by='model'` (the default) is where the signal is right
now.


In [5]:
plot_pca("NPF2.12_Q9LFX9")  # no clustering, colored by model (default) -- which of the 6 backends produced each point
plot_pca("NPF2.12_Q9LFX9", cluster_method="gmm", n_components="auto")


[reannotate] NPF2.12_Q9LFX9/pca_k3: 60 symlinks (max 20/cluster) of 4621 assignments -> ../results/tm_reannotated/NPF2.12_Q9LFX9/pca_k3
